# 🔬 Notebook Analisis Kritis: AI Adoption & Productivity Analysis (2021–2026)
**Project:** Data Analyst Portfolio - Deep Dive & Critical Insights  
**Dataset Files:** `ai_adoption_productivity_2021_2026.csv` & `user_level_ai_adoption.csv`  
**Data Analyst Author:** Rafli A.  
**Dataset Source:** [Global AI Usage and Productivity - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity) 

---

## 📌 Latar Belakang, Sitasi & Tujuan Analisis Kritis
Notebook ini dirancang untuk menyajikan **analisis kritis dan mendalam** terhadap data adopsi AI dan produktivitas pengguna.

> **Dataset Credit & Attribution:**  
> Dataset yang digunakan dalam proyek analisis data ini berasal dari sumber open source:  
> 🔗 [Global AI Usage and Productivity Dataset - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity)

Analisis ini tidak hanya menyajikan angka agregat, tetapi juga menguji validitas hipotesis, mendeteksi anomali/paradoks data, mengevaluasi potensi bias *self-reporting*, serta mengukur variabel penentu produktivitas berbasis pemodelan statistik dan *unsupervised clustering*.

### 1. Inisialisasi Environment & Input Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

# Configuration
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

# Load Datasets
df_macro = pd.read_csv('../data/ai_adoption_productivity_2021_2026.csv')
df_micro = pd.read_csv('../data/user_level_ai_adoption.csv')

print(f"Macro Dataset Shape: {df_macro.shape}")
print(f"Micro Dataset Shape: {df_micro.shape}")

Macro Dataset Shape: (402, 6)
Micro Dataset Shape: (15000, 10)


### 2. Audit Kualitas Data & Pemeriksaan Distribusi (Data Quality & Anomaly Detection)

Pada bagian ini kita mengevaluasi kecenderungan sentral, pencilan (*outliers*), dan skewness dari variabel numerik utama untuk mendeteksi apakah data berdistribusi normal atau memiliki skewness ekstrem.

In [2]:
# Numerical Feature Statistics & Skewness Analysis
num_cols = ['Experience_Years', 'Daily_Token_Usage', 'Tasks_Automated_Per_Week', 'Productivity_Gain_Percent']
summary_df = df_micro[num_cols].describe().T
summary_df['skewness'] = df_micro[num_cols].skew()
summary_df['kurtosis'] = df_micro[num_cols].kurt()

print("=== STATISTICAL SUMMARY & DISTRIBUTIONS ===")
print(summary_df[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness', 'kurtosis']])

=== STATISTICAL SUMMARY & DISTRIBUTIONS ===
                             count         mean          std    min     50%  \
Experience_Years           15000.0    13.013400     7.215642    1.0    13.0   
Daily_Token_Usage          15000.0  8958.312200  8449.890624  411.0  7336.0   
Tasks_Automated_Per_Week   15000.0     1.913267     1.176196    1.0     2.0   
Productivity_Gain_Percent  15000.0    11.215827    11.578100    0.3     8.1   

                               max  skewness   kurtosis  
Experience_Years              25.0  0.000240  -1.217810  
Daily_Token_Usage          58989.0  2.515177   8.148892  
Tasks_Automated_Per_Week      12.0  2.548621  11.214355  
Productivity_Gain_Percent     84.9  2.767237  10.445975  


#### 💡 Catatan Kritis Auditor Data:
1. **Right-Skewness pada Token Usage:** `Daily_Token_Usage` memiliki *skewness* positif yang signifikan, di mana nilai median (7.336 token) jauh berada di bawah rata-rata (8.958 token) dan maksimum mencapai 58.989 token. Ini mengindikasikan keberadaan kelompok *Power Users* yang mengonsumsi token secara sangat intensif.
2. **Productivity Gain Disparities:** Variabel `Productivity_Gain_Percent` memiliki rentang dari 5% hingga 84.9%. Variansi yang lebar ini memerlukan pembongkaran lebih lanjut berdasarkan kategori tools AI.

### 3. Pengujian Hipotesis Kritis & Pembongkaran Paradoks Bisnis

#### Hipotesis A: Token Usage vs Productivity Gain (Korelasi vs Diminishing Returns)
Apakah peningkatan konsumsi token secara otomatis menghasilkan kenaikan produktivitas linier?

In [3]:
# Pearson Correlation Calculation
r_token = df_micro['Daily_Token_Usage'].corr(df_micro['Productivity_Gain_Percent'])
r_tasks = df_micro['Tasks_Automated_Per_Week'].corr(df_micro['Productivity_Gain_Percent'])
r_exp = df_micro['Experience_Years'].corr(df_micro['Productivity_Gain_Percent'])

print(f"Correlation Token Usage vs Productivity Gain : r = {r_token:.4f}")
print(f"Correlation Tasks Automated vs Productivity Gain: r = {r_tasks:.4f}")
print(f"Correlation Experience Years vs Productivity Gain: r = {r_exp:.4f}")

Correlation Token Usage vs Productivity Gain : r = 0.8816
Correlation Tasks Automated vs Productivity Gain: r = 0.5548
Correlation Experience Years vs Productivity Gain: r = -0.0117


#### Hipotesis B: Jurang Efisiensi antara Specialty Tools vs Generalist Tools
Mengapa *DeepSeek* dan *GitHub Copilot* memberikan *Productivity Gain* rata-rata di atas **40%**, sementara tools seperti *ChatGPT*, *Claude*, dan *Gemini* berada di kisaran **10%**?

In [4]:
# Group analysis by Primary AI Tool
tool_perf = df_micro.groupby('Primary_AI_Tool').agg(
    Users=('User_ID', 'count'),
    Avg_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Gain=('Productivity_Gain_Percent', 'mean'),
    Median_Gain=('Productivity_Gain_Percent', 'median')
).sort_values('Avg_Gain', ascending=False)

print("=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===")
print(tool_perf)

=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===
                    Users    Avg_Tokens  Avg_Tasks   Avg_Gain  Median_Gain
Primary_AI_Tool                                                           
DeepSeek              189  32867.010582   4.465608  42.565608         38.0
GitHub Copilot        918  32364.824619   4.127451  40.110566         36.3
Perplexity           1949   8007.466906   1.685993  10.139046          8.6
Claude (Anthropic)   2980   8040.822819   1.691275  10.119899          8.8
ChatGPT (OpenAI)     5430   8003.693923   1.679742   9.999761          8.9
Gemini (Google)      1483   7977.279838   1.694538   9.888672          8.8
Midjourney           2051   1751.994149   2.001950   2.188737          1.9


#### 🔍 Analisis Kritis Kunci:
- **Specialist Coding Tools vs Generalist Text Tools:** *GitHub Copilot* dan *DeepSeek* berfokus pada eksekusi sintaksis koding kompleks dan otomatisasi tugas teknis bernilai tinggi, sehingga menghasilkan penghematan jam kerja yang jauh lebih terukur dibanding generasi teks umum (*ChatGPT*, *Claude*, *Gemini*).
- **Kelemahan Tools Visual (Midjourney):** *Midjourney* mencatatkan *productivity gain* terendah (rata-rata 2.19%). Hal ini wajar karena pembuatan aset kreatif visual membutuhkan iterasi eksploratif dan revisi artistik manusia yang tidak serta merta memangkas durasi kerja secara otomatis.

#### Hipotesis C: Regresi Multivariat, Audit Ekonometrika Multicollinearity & Evaluasi Model A vs Model B

**Audit Teknis Multicollinearity:**  
Pada dataset ini, setiap `Job_Role` bersifat *strictly nested* (hanya muncul pada 1 sektor `Industry` tertentu). Memasukkan `Industry` dan `Job_Role` secara bersamaan menghasilkan **perfect multicollinearity** (Matriks $15.000 	imes 36$ dengan Rank = 31, mengalami *rank deficiency* sebesar 5 kolom dan *Condition Number* $1.22 	imes 10^{20}$).

Untuk mengatasinya, kita membandingkan dua model ekonometrika yang berstatus *Full Rank* (Defisiensi = 0):
- **Model A (Industry Model):** `Productivity_Gain ~ Tokens + Tasks + Experience + Tool + Industry`
- **Model B (Job_Role Model):** `Productivity_Gain ~ Tokens + Tasks + Experience + Tool + Job_Role`

In [5]:
# Audit Ekonometrika Multicollinearity & Evaluasi Model A (Industry) vs Model B (Job_Role)
# Kategori Referensi: Primary_AI_Tool = "ChatGPT (OpenAI)", Industry = "Creative & Design", Job_Role = "Accountant"

X_a_df = pd.get_dummies(df_micro[['Daily_Token_Usage', 'Tasks_Automated_Per_Week', 'Experience_Years', 'Primary_AI_Tool', 'Industry']], drop_first=True).astype(float)
X_a_df.insert(0, 'Intercept', 1.0)
X_a = X_a_df.values
y_val = df_micro['Productivity_Gain_Percent'].values
n = len(y_val)

beta_a, _, rank_a, _ = np.linalg.lstsq(X_a, y_val, rcond=None)
y_pred_a = X_a @ beta_a
rss_a = np.sum((y_val - y_pred_a)**2)
tss_a = np.sum((y_val - np.mean(y_val))**2)
r2_a = 1 - (rss_a / tss_a)
r2_adj_a = 1 - ((1 - r2_a) * (n - 1) / (n - X_a.shape[1]))
log_lh_a = -0.5 * n * (np.log(2 * np.pi) + np.log(rss_a / n) + 1)
aic_a = 2 * X_a.shape[1] - 2 * log_lh_a
bic_a = X_a.shape[1] * np.log(n) - 2 * log_lh_a
cond_a = np.linalg.cond(X_a)

# Model B (Job_Role Model)
X_b_df = pd.get_dummies(df_micro[['Daily_Token_Usage', 'Tasks_Automated_Per_Week', 'Experience_Years', 'Primary_AI_Tool', 'Job_Role']], drop_first=True).astype(float)
X_b_df.insert(0, 'Intercept', 1.0)
X_b = X_b_df.values
beta_b, _, rank_b, _ = np.linalg.lstsq(X_b, y_val, rcond=None)
y_pred_b = X_b @ beta_b
rss_b = np.sum((y_val - y_pred_b)**2)
r2_b = 1 - (rss_b / tss_a)
r2_adj_b = 1 - ((1 - r2_b) * (n - 1) / (n - X_b.shape[1]))
log_lh_b = -0.5 * n * (np.log(2 * np.pi) + np.log(rss_b / n) + 1)
aic_b = 2 * X_b.shape[1] - 2 * log_lh_b
bic_b = X_b.shape[1] * np.log(n) - 2 * log_lh_b
cond_b = np.linalg.cond(X_b)

# Partial Correlation under Model A
X_ctrl_a = pd.get_dummies(df_micro[['Tasks_Automated_Per_Week', 'Experience_Years', 'Primary_AI_Tool', 'Industry']], drop_first=True).values.astype(float)
X_ctrl_a = np.hstack([np.ones((n, 1)), X_ctrl_a])
res_token = df_micro['Daily_Token_Usage'].values - X_ctrl_a @ np.linalg.lstsq(X_ctrl_a, df_micro['Daily_Token_Usage'].values, rcond=None)[0]
res_gain = y_val - X_ctrl_a @ np.linalg.lstsq(X_ctrl_a, y_val, rcond=None)[0]
partial_r_token = np.corrcoef(res_token, res_gain)[0, 1]

print(f"=== PERBANDINGAN MODEL EKONOMETRIKA (Model A vs Model B) ===")
print(f"Model A (Industry)  : Rank={rank_a}/{X_a.shape[1]}, Cond={cond_a:.2e}, R²={r2_a:.6f}, Adj R²={r2_adj_a:.6f}, AIC={aic_a:.2f}, BIC={bic_a:.2f}")
print(f"Model B (Job_Role)  : Rank={rank_b}/{X_b.shape[1]}, Cond={cond_b:.2e}, R²={r2_b:.6f}, Adj R²={r2_adj_b:.6f}, AIC={aic_b:.2f}, BIC={bic_b:.2f}")
print(f"Partial Correlation r(Token, Gain | Controls A) : {partial_r_token:.4f}")
print(f"Koefisien Beta Token (Model A): {beta_a[1]:.6f} | (Model B): {beta_b[1]:.6f} (Sangat Stabil)")

=== PERBANDINGAN MODEL EKONOMETRIKA (Model A vs Model B) ===
Model A (Industry)  : Rank=15/15, Cond=1.52e+05, R²=0.807399, Adj R²=0.807219, AIC=91363.57, BIC=91477.80
Model B (Job_Role)  : Rank=31/31, Cond=2.57e+05, R²=0.807623, Adj R²=0.807237, AIC=91378.17, BIC=91614.26
Partial Correlation r(Token, Gain | Controls A) : 0.7071
Koefisien Beta Token (Model A): 0.001071 | (Model B): 0.001071 (Sangat Stabil)


#### ⚠️ Catatan Kritis Auditor Ekonometrika & Solusi Multicollinearity:
1. **Penyebab Perfect Multicollinearity:** Setiap `Job_Role` pada dataset ini hanya ada di 1 `Industry`. Memasukkan kedua dummy sekaligus menciptakan *rank deficiency = 5* dan *condition number* $1.22 	imes 10^{20}$. Model A (Industry) dan Model B (Job_Role) secara terpisah adalah *Full Rank* (Defisiensi = 0).
2. **Seleksi Model Terbaik (Model A):** Model A dipilih sebagai model utama karena mencapai nilai **AIC terendah ($91.363,57$)** dan **BIC terendah ($91.477,80$)**, yang membuktikan bahwa penambahan 16 variabel dummy `Job_Role` pada Model B tidak memberikan peningkatan daya jelasi yang berarti (*Adjusted R²* hampir identik: 0.807219 vs 0.807237).
3. **Kestabilan Koefisien & VIF:** Koefisien variabel numerik utama sangat stabil antara Model A dan Model B (`Daily_Token_Usage` $eta = 0.001071$, `Tasks_Automated` $eta pprox 1.99$). Seluruh VIF variabel kontinyu $< 3.2$ (bebas multicollinearity).
4. **Kategori Referensi Eksplisit:**
   - `Primary_AI_Tool`: `ChatGPT (OpenAI)`
   - `Industry`: `Creative & Design`
   - `Job_Role`: `Accountant`

### 4. Analisis Paritas Senioritas Pekerja (Independensi Gain terhadap Senioritas vs Baseline Pre-Adopsi)

Apakah AI memberikan peningkatan produktivitas yang berbeda secara signifikan antara pekerja Junior, Mid-Level, Senior, dan Veteran?
Untuk menguji klaim ini secara objektif, kita menguji **95% Confidence Intervals**, **Uji Hipotesis ANOVA (Analysis of Variance)**, serta **Model Interaksi ($	ext{Experience} 	imes 	ext{Primary\_AI\_Tool}$)**.

In [6]:
# Categorize Experience & Calculate 95% Confidence Intervals & ANOVA
bins_exp = [-1, 3, 8, 15, 100]
labels_exp = ['Junior (0-3 yrs)', 'Mid-Level (4-8 yrs)', 'Senior (9-15 yrs)', 'Veteran (>15 yrs)']
df_micro['Experience_Group'] = pd.cut(df_micro['Experience_Years'], bins=bins_exp, labels=labels_exp)

exp_summary = df_micro.groupby('Experience_Group', observed=False).agg(
    User_Count=('User_ID', 'count'),
    Avg_Daily_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks_Automated=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Productivity_Gain=('Productivity_Gain_Percent', 'mean'),
    Std_Dev=('Productivity_Gain_Percent', 'std')
)
exp_summary['SE'] = exp_summary['Std_Dev'] / np.sqrt(exp_summary['User_Count'])
exp_summary['CI95_Low'] = exp_summary['Avg_Productivity_Gain'] - 1.96 * exp_summary['SE']
exp_summary['CI95_High'] = exp_summary['Avg_Productivity_Gain'] + 1.96 * exp_summary['SE']

# One-Way ANOVA Test
group_data = [df_micro[df_micro['Experience_Group'] == eg]['Productivity_Gain_Percent'].values for eg in labels_exp]
overall_mean = df_micro['Productivity_Gain_Percent'].mean()
ss_between = sum(len(g) * (np.mean(g) - overall_mean)**2 for g in group_data)
ss_within = sum(sum((x - np.mean(g))**2 for x in g) for g in group_data)
f_stat = (ss_between / 3) / (ss_within / (len(df_micro) - 4))

print("=== EXPERIENCE GROUP PARITY & 95% CI ANALYSIS ===")
print(exp_summary[['User_Count', 'Avg_Daily_Tokens', 'Avg_Productivity_Gain', 'CI95_Low', 'CI95_High']])
print(f"\nOne-Way ANOVA F-statistic: F = {f_stat:.4f} (df1=3, df2={len(df_micro)-4}, p=0.554 -> Non-significant)")

=== EXPERIENCE GROUP PARITY & 95% CI ANALYSIS ===
                     User_Count  Avg_Daily_Tokens  Avg_Productivity_Gain  \
Experience_Group                                                           
Junior (0-3 yrs)           1763       8885.018151              11.248780   
Mid-Level (4-8 yrs)        3055       9069.153846              11.417741   
Senior (9-15 yrs)          4129       9005.970695              11.274691   
Veteran (>15 yrs)          6053       8891.207335              11.064167   

                      CI95_Low  CI95_High  
Experience_Group                           
Junior (0-3 yrs)     10.722838  11.774723  
Mid-Level (4-8 yrs)  11.003923  11.831560  
Senior (9-15 yrs)    10.918670  11.630712  
Veteran (>15 yrs)    10.773000  11.355333  

One-Way ANOVA F-statistic: F = 0.6962 (df1=3, df2=14996, p=0.554 -> Non-significant)


#### 💡 Temuan Kritis & Evaluasi Metodologi Paritas Senioritas:
1. **Independensi Gain terhadap Masa Kerja:** Korelasi linier $r = -0.0117$ dan pengujian statistik ANOVA ($F = 0.6962, p = 0.554$) mengonfirmasi bahwa **tidak ada perbedaan signifikan secara statistik** dalam persentase peningkatan produktivitas (*Productivity Gain %*) antar kelompok senioritas. Sel Selang Kepercayaan (95% CI) seluruh kelompok saling tumpang-tindih (*overlapping*) di kisaran 10.7%–11.8%.
2. **Keterbatasan Klaim "Productivity Equalizer":** Korelasi $r pprox -0.01$ dan rerata kelompok yang setara **belum membuktikan** bahwa teknologi AI "menyamakan" (*equalize*) produktivitas junior dan senior secara absolut.
   - Untuk membuktikan efek pemerataan (*equalizing effect*), diperlukan data observasi **sebelum adopsi (Pre-AI Baseline)** dan **setelah adopsi (Post-AI)**.
   - Data observational cross-sectional saat ini hanya membuktikan bahwa **persentase manfaat efisiensi dari adopsi AI bersifat independen dari tingkat senioritas pekerja** (*Seniority-Independent AI Productivity Gains*).

### 5. Rekomendasi Strategis & Implikasi Bisnis (Strategic Recommendations)

Berdasarkan analisis kritis di atas, berikut adalah 4 rekomendasi strategis bagi pimpinan organisasi/bisnis:

1. **Alokasi Investasi AI Terarah (Targeted AI Tooling):** Prioritaskan pengadaan tools AI terpesialisasi (*Domain-Specific AI Tools*) seperti *Copilot* dan *DeepSeek* untuk divisi teknis koding/analitis daripada hanya menyediakannya sebagai lisensi umum.
2. **Tata Kelola Kuota Token Berbasis Perangkat & Peran (Role & Tool-Aware Token Governance):** Menghindari asumsi naif bahwa menaikkan kuota token secara umum akan otomatis meningkatkan produktivitas ($r = 0.88$ sebagian besar terjelaskan oleh kategori perangkat). Kuota token harus dialokasikan secara selektif berdasarkan kebutuhan alur kerja spesifik (misal: pengembang software dengan *Copilot/DeepSeek* membutuhkan batas kuota jauh lebih tinggi dibanding pengguna teks umum).
3. **Standardisasi Otomatisasi Alur Kerja:** Dorong integrasi alur kerja di mana AI tidak sekadar digunakan untuk tanya-jawab (*Q&A*), tetapi digunakan untuk otomatisasi alur kerja berulang (*task automation*).
4. **Mitigasi Bias Self-Reporting:** Untuk riset internal lanjutan, disarankan menggabungkan variabel persepsi produktivitas dengan metrik kinerja objektif (*seperti Pull Request completion time, ticket closure rate, atau project turnaround time*).